In [11]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# creating fix length waveform dataset

In [12]:
MAX_LENGTH = 48000

# finding metedata path 

In [13]:
def load_audio(file_path):
    
    y, sr = librosa.load(file_path, sr=16000)
    
    y = librosa.util.normalize(y)
    
    if len(y) > MAX_LENGTH:
        y = y[:MAX_LENGTH]
        
    else:
        padding = MAX_LENGTH - len(y)
        y = np.pad(y, (0, padding))
    
    return y

In [14]:
metadata_path = '/kaggle/input/datasets/hahunavth/asvspoof2019-la/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt'

metadata = pd.read_csv(
    metadata_path,
    sep=' ',
    header=None,
    names=['speaker','file','system','null','label']
)

metadata.head()

,speaker,file,system,null,label
0,LA_0079,LA_T_1138215,-,-,bonafide
1,LA_0079,LA_T_1271820,-,-,bonafide
2,LA_0079,LA_T_1272637,-,-,bonafide
3,LA_0079,LA_T_1276960,-,-,bonafide
4,LA_0079,LA_T_1341447,-,-,bonafide


creating proper audio paths 

In [15]:
base_dir = '/kaggle/input/datasets/hahunavth/asvspoof2019-la/LA/ASVspoof2019_LA_train/flac/'

In [16]:
metadata['full_path'] = metadata['file'].apply(
    lambda x: base_dir + x + '.flac'
)

In [17]:
print(metadata['full_path'].iloc[0])

/kaggle/input/datasets/hahunavth/asvspoof2019-la/LA/ASVspoof2019_LA_train/flac/LA_T_1138215.flac


In [18]:
import os

os.path.exists(metadata['full_path'].iloc[0])

True

In [19]:
print(metadata['label'].value_counts())

label
spoof       22800
bonafide     2580
Name: count, dtype: int64


# building waveform dataset

In [20]:
import librosa
import numpy as np

def load_audio(file_path):
    # 1. Load at 16kHz and convert to mono channel [cite: 173, 211]
    y, sr = librosa.load(file_path, sr=16000, mono=True)
    
    # 2. Normalize amplitude (zero mean, unit variance / max absolute) [cite: 181, 213]
    if np.max(np.abs(y)) > 0:
        y = y / np.max(np.abs(y))
    
    # 3. Fixed 3-second input (48,000 samples) [cite: 182, 217]
    target_samples = 48000
    if len(y) > target_samples:
        y = y[:target_samples] # Trim [cite: 217]
    else:
        y = np.pad(y, (0, target_samples - len(y)), mode='constant') # Zero-pad [cite: 217]
        
    return y

In [21]:
sample_waveform = load_audio(metadata['full_path'].iloc[0])

print(sample_waveform.shape)


(48000,)


In [35]:
sample_metadata = metadata.sample(10000, random_state=42)

# extracting waveforms and levels 

In [23]:
X = []
y = []

for _, row in sample_metadata.iterrows():
    
    try:
        waveform = load_audio(row['full_path'])
        
        X.append(waveform)
        y.append(row['label'])
        
    except:
        pass

In [24]:
import numpy as np

X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)

(25380, 48000)
(25380,)


In [25]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

y = encoder.fit_transform(y)

print(y[:10])

[0 0 0 0 0 0 0 0 0 0]


# reshaping for cnn model 

In [26]:
X = X.reshape(X.shape[0], X.shape[1], 1)

print(X.shape)

(25380, 48000, 1)


# train/test split 

In [27]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(20304, 48000, 1)
(5076, 48000, 1)


# build 1D CNN Model 

In [28]:
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    MaxPooling1D,
    GlobalAveragePooling1D,
    Dense,
    Dropout,
    BatchNormalization
)

from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Sequential

model = Sequential()


model.add(Input(shape=(48000,1)))

model.add(Conv1D(32,5,padding='same',activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling1D(2))

model.add(Conv1D(64,5,padding='same',activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling1D(2))

model.add(Conv1D(128,5,padding='same',activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling1D(2))

model.add(GlobalAveragePooling1D())

model.add(Dense(128,activation='relu'))

model.add(Dropout(0.5))

model.add(Dense(1,activation='sigmoid'))

# compiling model 

In [29]:
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [31]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(class_weights))

print(class_weights)

{0: np.float64(4.9186046511627906), 1: np.float64(0.5565789473684211)}


# train model 

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test,y_test),
    epochs=15,
    batch_size=64,
    callbacks=[early_stop],
    class_weight=class_weights
)

Epoch 1/15
318/318 ━━━━━━━━━━━━━━━━━━━━ 2904s 9s/step - accuracy: 0.6932 - loss: 0.5076 - val_accuracy: 0.8983 - val_loss: 0.2799
Epoch 2/15
 92/318 ━━━━━━━━━━━━━━━━━━━━ 33:06 9s/step - accuracy: 0.7131 - loss: 0.4602

# evaluate model 

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test)

print("Test Accuracy:", test_accuracy)

# make predictions 

In [ ]:
y_pred = model.predict(X_test)

y_pred = (y_pred > 0.5).astype(int)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

# confusion matrix 

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Raw Waveform CNN Confusion Matrix")

plt.show()

# plotting training curves 

In [ ]:
plt.figure(figsize=(10,4))

plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])

plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.legend(['Train', 'Validation'])

plt.show()